# SDC-2023 Multipath — Data Parser

Parses the Google Smartphone Decimeter Challenge **2023** training logs and uses each device's own `MultipathIndicator` field (0 = clean / not-flagged, 1 = multipath detected) as the ground-truth label — exactly the same label source used in the Mi8 pipeline. Because the label is reported per raw measurement by the receiver, **no SPAN time-sync is required**.

Only three device models in this dataset actually populate `MultipathIndicator = 1` (**pixel6pro, pixel7pro, sm-g955f**); every other device reports a constant 0, which means *not-populated* rather than *confirmed clean*. Including those would poison the clean class with unknown-status measurements, so this notebook **keeps only the device files that report at least one multipath flag**.

**Input:** `data/01_raw/sdc2023/train/2023-*/<device>/device_gnss.csv`

**Output:** `data/02_interim/sdc2023_epochs.csv`

## 1. Setup & Configuration

In [1]:
import os
import glob
import pandas as pd
import numpy as np

BASE_DIR   = os.path.abspath(os.path.join(os.getcwd(), '../..'))
RAW_DIR    = os.path.join(BASE_DIR, 'data/01_raw/sdc2023/train')
OUTPUT_DIR = os.path.join(BASE_DIR, 'data/02_interim')
OUTPUT_CSV = os.path.join(OUTPUT_DIR, 'sdc2023_epochs.csv')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Every device_gnss.csv under a 2023 session
GNSS_FILES = sorted(glob.glob(os.path.join(RAW_DIR, '2023-*', '*', 'device_gnss.csv')))
print(f'Found {len(GNSS_FILES)} device_gnss.csv files under 2023 sessions')

Found 54 device_gnss.csv files under 2023 sessions


## 2. Select the Multipath-Reporting Device Files

Scan every 2023 device file and keep only those whose `MultipathIndicator` column contains at least one `1`. These are the only files where the label is a genuine hardware determination rather than a constant placeholder.

In [2]:
def session_device(path):
    parts = path.replace('\\', '/').split('/')
    return parts[-3], parts[-2]   # session folder, device folder

selected = []
print(f'{"session":38} {"device":16} {"rows":>8} {"MP=1":>7}')
for f in GNSS_FILES:
    sess, dev = session_device(f)
    mp = pd.read_csv(f, usecols=['MultipathIndicator'])['MultipathIndicator']
    n1 = int((mp == 1).sum())
    if n1 > 0:
        selected.append((f, sess, dev))
        print(f'{sess:38} {dev:16} {len(mp):>8} {n1:>7}')

print(f'\nSelected {len(selected)} device files '
      f'across {len({s for _, s, _ in selected})} unique sessions.')

session                                device               rows    MP=1


2023-03-08-21-34-us-ca-mtv-u           pixel6pro           35446    4070
2023-03-08-21-34-us-ca-mtv-u           pixel7pro           37582    2360


2023-05-09-21-32-us-ca-mtv-pe1         pixel7pro           67128    3998


2023-05-16-19-55-us-ca-mtv-xe1         pixel7pro           63320    3955


2023-05-19-20-10-us-ca-mtv-ie2         pixel7pro           33093    1994


...

2023-09-06-22-49-us-ca-routebb1        pixel7pro           71974    4084


2023-09-07-18-59-us-ca                 pixel7pro           43156    2213


2023-09-07-19-33-us-ca                 pixel6pro           43205    5490
2023-09-07-19-33-us-ca                 sm-g955f            28643    1407


2023-09-07-22-47-us-ca-routebc2        pixel6pro           94247   11337



Selected 20 device files across 17 unique sessions.


## 3. Parse Raw GNSS Measurements

The 2023 `device_gnss.csv` is already a clean CSV (unlike the raw Mi8 `GnssLog.txt`), so parsing is a direct read. Alongside the signal-quality features shared with the Mi8 pipeline, the 2023 format also provides **satellite geometry** (elevation / azimuth), the **raw pseudorange**, and the columns needed to compute a **pseudorange residual** — all strong physical multipath indicators.

In [3]:
# Signal-quality + geometry features we keep as-is
FEATURE_COLS = [
    'utcTimeMillis', 'TimeNanos', 'FullBiasNanos', 'State', 'Svid',
    'ReceivedSvTimeUncertaintyNanos',
    'Cn0DbHz', 'BasebandCn0DbHz', 'SnrInDb', 'AgcDb',
    'PseudorangeRateMetersPerSecond', 'PseudorangeRateUncertaintyMetersPerSecond',
    'AccumulatedDeltaRangeState', 'AccumulatedDeltaRangeMeters',
    'AccumulatedDeltaRangeUncertaintyMeters',
    'CarrierFrequencyHz', 'ConstellationType', 'MultipathIndicator',
    'SvElevationDegrees', 'SvAzimuthDegrees',
    'RawPseudorangeMeters', 'RawPseudorangeUncertaintyMeters',
]
# Extra columns used ONLY to derive the pseudorange residual, then dropped
RESIDUAL_INPUTS = [
    'SvPositionXEcefMeters', 'SvPositionYEcefMeters', 'SvPositionZEcefMeters',
    'WlsPositionXEcefMeters', 'WlsPositionYEcefMeters', 'WlsPositionZEcefMeters',
    'SvClockBiasMeters', 'IsrbMeters',
    'IonosphericDelayMeters', 'TroposphericDelayMeters',
]

def parse_gnss(path, sess, dev):
    df = pd.read_csv(path, low_memory=False)
    keep = [c for c in FEATURE_COLS + RESIDUAL_INPUTS if c in df.columns]
    df = df[keep].copy()
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df['session'] = sess
    df['device']  = dev
    return df

frames = []
for f, sess, dev in selected:
    df = parse_gnss(f, sess, dev)
    frames.append(df)
    print(f'  {sess} / {dev}: {len(df):,} rows')

raw_df = pd.concat(frames, ignore_index=True)
print(f'\nTotal raw measurements: {len(raw_df):,}')
raw_df.head()

  2023-03-08-21-34-us-ca-mtv-u / pixel6pro: 35,446 rows


  2023-03-08-21-34-us-ca-mtv-u / pixel7pro: 37,582 rows


  2023-05-09-21-32-us-ca-mtv-pe1 / pixel7pro: 67,128 rows


  2023-05-16-19-55-us-ca-mtv-xe1 / pixel7pro: 63,320 rows


  2023-05-19-20-10-us-ca-mtv-ie2 / pixel7pro: 33,093 rows


...

  2023-09-06-22-49-us-ca-routebb1 / pixel7pro: 71,974 rows


  2023-09-07-18-59-us-ca / pixel7pro: 43,156 rows


  2023-09-07-19-33-us-ca / pixel6pro: 43,205 rows
  2023-09-07-19-33-us-ca / sm-g955f: 28,643 rows


  2023-09-07-22-47-us-ca-routebc2 / pixel6pro: 94,247 rows

Total raw measurements: 1,076,002


,utcTimeMillis,TimeNanos,FullBiasNanos,State,Svid,ReceivedSvTimeUncertaintyNanos,Cn0DbHz,BasebandCn0DbHz,SnrInDb,AgcDb,...,SvPositionZEcefMeters,WlsPositionXEcefMeters,WlsPositionYEcefMeters,WlsPositionZEcefMeters,SvClockBiasMeters,IsrbMeters,IonosphericDelayMeters,TroposphericDelayMeters,session,device
0,1678311289000,70135000000,-1362346436865248129,16431,6,14,41.710907,NaN,NaN,NaN,...,8.179664e+06,-2.691861e+06,-4.301447e+06,3.851258e+06,173771.573315,0.0,10.939039,3.068689,2023-03-08-21-34-us-ca-mtv-u,pixel6pro
1,1678311289000,70135000000,-1362346436865248129,16431,11,50,26.527052,NaN,NaN,NaN,...,-5.633358e+06,-2.691861e+06,-4.301447e+06,3.851258e+06,-37354.587769,0.0,17.747940,5.294300,2023-03-08-21-34-us-ca-mtv-u,pixel6pro
2,1678311289000,70135000000,-1362346436865248129,16431,12,27,34.177910,NaN,NaN,NaN,...,1.691337e+07,-2.691861e+06,-4.301447e+06,3.851258e+06,-104664.470842,0.0,17.929485,6.339256,2023-03-08-21-34-us-ca-mtv-u,pixel6pro
3,1678311289000,70135000000,-1362346436865248129,16396,13,1000000000,18.964769,NaN,NaN,NaN,...,NaN,-2.691861e+06,-4.301447e+06,3.851258e+06,NaN,NaN,NaN,NaN,2023-03-08-21-34-us-ca-mtv-u,pixel6pro
4,1678311289000,70135000000,-1362346436865248129,16384,14,1000000000,26.938070,NaN,NaN,NaN,...,NaN,-2.691861e+06,-4.301447e+06,3.851258e+06,NaN,NaN,NaN,NaN,2023-03-08-21-34-us-ca-mtv-u,pixel6pro


## 4. Outlier Rejection

Applies Google's official quality filters (identical to the Mi8 pipeline): valid non-zero `FullBiasNanos`, positive `TimeNanos`, a decoded/known time-of-week `State` bit, and code-lock timing uncertainty ≤ 500 ns.

In [4]:
def apply_outlier_rejection(df):
    n0 = len(df)
    df = df.dropna(subset=['FullBiasNanos', 'TimeNanos'])
    df = df[(df['FullBiasNanos'] != 0) & (df['TimeNanos'] > 0)]
    state = df['State'].fillna(0).astype('int64')
    state_ok = ((state & (1 << 3)) != 0) | ((state & (1 << 14)) != 0)
    df = df[state_ok]
    df = df[df['ReceivedSvTimeUncertaintyNanos'] <= 500]
    print(f'Rejected {n0 - len(df):,} rows  |  Kept {len(df):,} rows')
    return df

clean_df = apply_outlier_rejection(raw_df.copy())
clean_df.head()

Rejected 314,405 rows  |  Kept 761,597 rows


,utcTimeMillis,TimeNanos,FullBiasNanos,State,Svid,ReceivedSvTimeUncertaintyNanos,Cn0DbHz,BasebandCn0DbHz,SnrInDb,AgcDb,...,SvPositionZEcefMeters,WlsPositionXEcefMeters,WlsPositionYEcefMeters,WlsPositionZEcefMeters,SvClockBiasMeters,IsrbMeters,IonosphericDelayMeters,TroposphericDelayMeters,session,device
0,1678311289000,70135000000,-1362346436865248129,16431,6,14,41.710907,NaN,NaN,NaN,...,8.179664e+06,-2.691861e+06,-4.301447e+06,3.851258e+06,173771.573315,0.0,10.939039,3.068689,2023-03-08-21-34-us-ca-mtv-u,pixel6pro
1,1678311289000,70135000000,-1362346436865248129,16431,11,50,26.527052,NaN,NaN,NaN,...,-5.633358e+06,-2.691861e+06,-4.301447e+06,3.851258e+06,-37354.587769,0.0,17.747940,5.294300,2023-03-08-21-34-us-ca-mtv-u,pixel6pro
2,1678311289000,70135000000,-1362346436865248129,16431,12,27,34.177910,NaN,NaN,NaN,...,1.691337e+07,-2.691861e+06,-4.301447e+06,3.851258e+06,-104664.470842,0.0,17.929485,6.339256,2023-03-08-21-34-us-ca-mtv-u,pixel6pro
5,1678311289000,70135000000,-1362346436865248129,16431,17,14,42.474239,NaN,NaN,NaN,...,2.204581e+07,-2.691861e+06,-4.301447e+06,3.851258e+06,211169.641548,0.0,13.549537,4.172994,2023-03-08-21-34-us-ca-mtv-u,pixel6pro
6,1678311289000,70135000000,-1362346436865248129,16431,19,16,40.274414,NaN,NaN,NaN,...,2.067795e+07,-2.691861e+06,-4.301447e+06,3.851258e+06,86179.585461,0.0,9.502725,2.706804,2023-03-08-21-34-us-ca-mtv-u,pixel6pro


## 5. Compute GPS Time of Week

Derives `GpsTimeNanos` (nanoseconds since the start of the current GPS week) from `TimeNanos` and `FullBiasNanos`, giving an absolute time reference for ordering and per-epoch grouping.

In [5]:
NANOS_PER_SECOND = 1e9
SECONDS_PER_WEEK = 604800

raw_gps_ns = clean_df['TimeNanos'].astype('int64') - clean_df['FullBiasNanos'].astype('int64')
clean_df['GpsTimeNanos'] = raw_gps_ns % int(SECONDS_PER_WEEK * NANOS_PER_SECOND)
print('GpsTimeNanos range:'
      f"  min={clean_df['GpsTimeNanos'].min():,}  max={clean_df['GpsTimeNanos'].max():,}")
clean_df[['session', 'device', 'GpsTimeNanos', 'Cn0DbHz', 'SvElevationDegrees', 'MultipathIndicator']].head()

GpsTimeNanos range:  min=244,519,000,491,899  max=505,900,000,263,556


,session,device,GpsTimeNanos,Cn0DbHz,SvElevationDegrees,MultipathIndicator
0,2023-03-08-21-34-us-ca-mtv-u,pixel6pro,336907000248129,41.710907,54.479871,0
1,2023-03-08-21-34-us-ca-mtv-u,pixel6pro,336907000248129,26.527052,28.062975,0
2,2023-03-08-21-34-us-ca-mtv-u,pixel6pro,336907000248129,34.177910,23.090938,0
5,2023-03-08-21-34-us-ca-mtv-u,pixel6pro,336907000248129,42.474239,36.711941,0
6,2023-03-08-21-34-us-ca-mtv-u,pixel6pro,336907000248129,40.274414,67.378019,0


## 6. Compute Pseudorange Residual

The single most physically-direct multipath signature is the **pseudorange residual** — the difference between the measured raw pseudorange and the geometric range implied by the satellite and receiver positions.

For each measurement:

```
geometric_range = || SvPosition_ECEF  -  WlsPosition_ECEF ||
modeled_range   = geometric_range  - SvClockBias  + Iono + Tropo + Isrb
residual_raw    = RawPseudorange    - modeled_range
```

`residual_raw` still contains the receiver clock bias, which is **common to every satellite in the same epoch**. Subtracting the per-epoch median removes it, leaving the per-satellite error — dominated by multipath and thermal noise. A large `|prResidual|` is a classic multipath tell.

In [6]:
geo_cols_present = all(c in clean_df.columns for c in [
    'SvPositionXEcefMeters', 'WlsPositionXEcefMeters', 'RawPseudorangeMeters'])

if geo_cols_present:
    sv = clean_df[['SvPositionXEcefMeters', 'SvPositionYEcefMeters', 'SvPositionZEcefMeters']].to_numpy()
    rx = clean_df[['WlsPositionXEcefMeters', 'WlsPositionYEcefMeters', 'WlsPositionZEcefMeters']].to_numpy()
    geo = np.sqrt(((sv - rx) ** 2).sum(axis=1))
    modeled = (geo
               - clean_df['SvClockBiasMeters'].fillna(0)
               + clean_df['IonosphericDelayMeters'].fillna(0)
               + clean_df['TroposphericDelayMeters'].fillna(0)
               + clean_df['IsrbMeters'].fillna(0))
    res_raw = clean_df['RawPseudorangeMeters'] - modeled
    clean_df['_res_raw'] = res_raw
    # de-mean the receiver clock bias per (session, device, epoch)
    epoch_median = clean_df.groupby(['session', 'device', 'utcTimeMillis'])['_res_raw'].transform('median')
    clean_df['prResidual'] = clean_df['_res_raw'] - epoch_median
    clean_df.drop(columns=['_res_raw'], inplace=True)
    print('prResidual computed.')
    print(clean_df['prResidual'].describe())
else:
    print('Geometry columns missing — skipping prResidual.')

# Drop the raw geometry/model inputs now that the residual is derived
clean_df.drop(columns=[c for c in RESIDUAL_INPUTS if c in clean_df.columns], inplace=True)

prResidual computed.
count    721832.000000
mean        -34.379787
std        3515.576446
min     -296024.948632
25%         -16.385391
50%           0.000000
75%          14.575824
max       40262.703553
Name: prResidual, dtype: float64


## 7. Label Distribution

How much multipath was flagged overall and per session/device.

In [7]:
total = len(clean_df)
dist  = clean_df['MultipathIndicator'].value_counts().sort_index()
print('=== Overall MultipathIndicator distribution ===')
for val, cnt in dist.items():
    print(f'  {int(val)} : {cnt:>9,}  ({100*cnt/total:.1f}%)')

print('\n=== Per session / device ===')
grp = (clean_df.groupby(['session', 'device', 'MultipathIndicator']).size()
       .unstack(fill_value=0).rename(columns={0: 'Clean', 1: 'Multipath'}))
if 'Multipath' in grp.columns:
    grp['Multipath_%'] = (100 * grp['Multipath'] / grp.sum(axis=1)).round(1)
print(grp)

=== Overall MultipathIndicator distribution ===
  0 :   708,154  (93.0%)
  1 :    53,443  (7.0%)

=== Per session / device ===


MultipathIndicator                         Clean  Multipath  Multipath_%
session                         device                                  
2023-03-08-21-34-us-ca-mtv-u    pixel6pro  23004       2939         11.3
                                pixel7pro  27047       1704          5.9
2023-05-09-21-32-us-ca-mtv-pe1  pixel7pro  46040       2117          4.4
2023-05-16-19-55-us-ca-mtv-xe1  pixel7pro  40396       2081          4.9
2023-05-19-20-10-us-ca-mtv-ie2  pixel7pro  21311        936          4.2
2023-05-24-20-26-us-ca-sjc-ge2  pixel7pro  31057       2836          8.4
2023-05-25-19-10-us-ca-sjc-be2  pixel7pro  28804       1141          3.8
2023-05-25-20-11-us-ca-sjc-he2  pixel7pro  27969       1183          4.1
2023-05-26-18-51-us-ca-sjc-ge2  pixel7pro  29132       2083          6.7
2023-09-05-20-13-us-ca          pixel7pro  49760       1828          3.5
2023-09-05-23-07-us-ca-routen   pixel7pro  46671       6660         12.5
2023-09-06-00-01-us-ca-routen   pixel6pro  47353   

## 8. Export

Drops the raw time columns that are not predictive features and writes the interim epoch table.

In [8]:
final_df = clean_df.drop(columns=['TimeNanos', 'FullBiasNanos', 'utcTimeMillis'], errors='ignore').copy()
final_df.dropna(subset=['MultipathIndicator'], inplace=True)
final_df['MultipathIndicator'] = final_df['MultipathIndicator'].astype(int)

final_df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {len(final_df):,} rows  ->  {OUTPUT_CSV}')
print(f'Columns: {list(final_df.columns)}')
final_df.head()

Saved 761,597 rows  ->  C:\Users\Dell\Documents\Warwick\Diss\Code\Dissertation_Trial\GNSS_Multipath_Project\data/02_interim\sdc2023_epochs.csv
Columns: ['State', 'Svid', 'ReceivedSvTimeUncertaintyNanos', 'Cn0DbHz', 'BasebandCn0DbHz', 'SnrInDb', 'AgcDb', 'PseudorangeRateMetersPerSecond', 'PseudorangeRateUncertaintyMetersPerSecond', 'AccumulatedDeltaRangeState', 'AccumulatedDeltaRangeMeters', 'AccumulatedDeltaRangeUncertaintyMeters', 'CarrierFrequencyHz', 'ConstellationType', 'MultipathIndicator', 'SvElevationDegrees', 'SvAzimuthDegrees', 'RawPseudorangeMeters', 'RawPseudorangeUncertaintyMeters', 'session', 'device', 'GpsTimeNanos', 'prResidual']


,State,Svid,ReceivedSvTimeUncertaintyNanos,Cn0DbHz,BasebandCn0DbHz,SnrInDb,AgcDb,PseudorangeRateMetersPerSecond,PseudorangeRateUncertaintyMetersPerSecond,AccumulatedDeltaRangeState,...,ConstellationType,MultipathIndicator,SvElevationDegrees,SvAzimuthDegrees,RawPseudorangeMeters,RawPseudorangeUncertaintyMeters,session,device,GpsTimeNanos,prResidual
0,16431,6,14,41.710907,NaN,NaN,NaN,-164.962361,0.150000,25,...,1,0,54.479871,128.729618,2.090664e+07,4.197094,2023-03-08-21-34-us-ca-mtv-u,pixel6pro,336907000248129,-17.517512
1,16431,11,50,26.527052,NaN,NaN,NaN,-534.366362,0.940767,29,...,1,0,28.062975,175.168830,2.299924e+07,14.989623,2023-03-08-21-34-us-ca-mtv-u,pixel6pro,336907000248129,20.691331
2,16431,12,27,34.177910,NaN,NaN,NaN,-402.783540,0.596536,25,...,1,0,23.090938,296.403147,2.329942e+07,8.094396,2023-03-08-21-34-us-ca-mtv-u,pixel6pro,336907000248129,20.895644
5,16431,17,14,42.474239,NaN,NaN,NaN,616.955107,0.150000,25,...,1,0,36.711941,46.197646,2.242312e+07,4.197094,2023-03-08-21-34-us-ca-mtv-u,pixel6pro,336907000248129,-19.880819
6,16431,19,16,40.274414,NaN,NaN,NaN,350.406984,0.150000,25,...,1,0,67.378019,30.702359,2.035373e+07,4.796679,2023-03-08-21-34-us-ca-mtv-u,pixel6pro,336907000248129,-5.631828
